# Module 6.3: Reward Models & RLHF

**Module 6.1 (SFT)** taught the model a *format*: given a user turn, produce an
assistant turn. Its own summary was blunt about the limit — SFT "teaches it the
format, not *values*." A model can follow ChatML perfectly and still be rude,
unhelpful, evasive, or confidently wrong.

So we need to train on **quality**. And that runs straight into a wall:

> **You cannot write down the right answer.**

Supervised learning needs a target. For "explain recursion to a beginner" there is no
single correct string — there are millions of good answers and billions of bad ones,
and the difference is a matter of judgement. Cross-entropy has nothing to bite on.

But there is something humans *can* do reliably, cheaply, and consistently:

> **Shown two answers, say which one is better.**

That single shift — from *"write the ideal answer"* to *"pick the better of two"* — is
the entire foundation of modern alignment. This module builds the machinery that turns
those comparisons into a training signal:

```mermaid
flowchart LR
    A["preference pairs<br/>(prompt, chosen, rejected)"] --> B["reward model<br/>r(x, y) -> a number"]
    B --> C["optimize the policy<br/>to score high on r"]
    C --> D["...without drifting<br/>too far from where<br/>it started (KL)"]
```

### Why this module exists (and why it comes before DPO)

**Module 6.4 is DPO**, and DPO is usually introduced as "the simple alternative to
RLHF." That framing is only useful if you know what RLHF *is* — and the DPO loss you'll
meet next module is not an independent invention. It is a **mathematical rearrangement
of the objective built in this notebook**, with the reward model solved away
analytically.

Two symbols in the DPO loss, $\beta$ and $\pi_{\text{ref}}$, come directly from here.
Meet them now, in the setting that gave rise to them, and DPO reads as a clever
simplification instead of an incantation.

**Prerequisites:** 5.1 (cross-entropy), 5.3 (you have a trained checkpoint), 6.1 (SFT).

**Papers this unlocks:** *Deep RL from Human Preferences* (Christiano et al., 2017),
*InstructGPT* (Ouyang et al., 2022), *Constitutional AI*, and the reasoning-model RL
literature (GRPO and friends) that all of 2024–25 is built on.

## 1. Turning comparisons into numbers: Bradley–Terry

We have a pile of human judgements of the form *"answer A beats answer B"*. We want a
function $r(x, y)$ — a **reward model** — that assigns each answer a *score*, such that
better answers score higher.

This problem is older than deep learning. It's the same problem as ranking chess
players from match outcomes, and it has a classic answer: the **Bradley–Terry model**.

Bradley–Terry says: if answer $y_w$ has score $r_w$ and answer $y_l$ has score $r_l$,
then the probability a human prefers $y_w$ is

$$P(y_w \succ y_l) = \frac{e^{r_w}}{e^{r_w} + e^{r_l}} = \sigma(r_w - r_l)$$

where $\sigma$ is the **sigmoid**. Read the middle expression: it is exactly a
**softmax over two options** (Module 1.2). And the right-hand form says something
important — the probability depends *only on the difference* $r_w - r_l$, never on the
absolute values.

> **Consequence worth remembering:** reward-model scores have **no absolute meaning**.
> Add 100 to every score and every prediction is unchanged. Only gaps matter. This is
> why you'll never see a "good answers score above 7.0" rule — the scale is arbitrary.

To fit it, we do what we always do: maximize the likelihood of what humans actually
chose, i.e. minimize its negative log:

$$\mathcal{L}_{\text{RM}} = -\log \sigma\big(r(x, y_w) - r(x, y_l)\big)$$

That is the **entire** reward-model loss. One line. Let's read its shape before we
train anything.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# The loss as a function of the score GAP (chosen minus rejected).
gap = torch.linspace(-6, 6, 200)
loss = -F.logsigmoid(gap)

print(f"{'gap (r_chosen - r_rejected)':>30}{'P(human picks chosen)':>24}{'loss':>10}")
print("-" * 66)
for g in [-4.0, -1.0, 0.0, 1.0, 4.0]:
    t = torch.tensor(g)
    print(f"{g:>30.1f}{torch.sigmoid(t).item():>24.3f}{-F.logsigmoid(t).item():>10.3f}")

plt.figure(figsize=(6, 3.5))
plt.plot(gap, loss)
plt.axvline(0, color="grey", ls="--", lw=1)
plt.xlabel("r(chosen) - r(rejected)"); plt.ylabel("loss")
plt.title("Bradley-Terry loss: push the gap positive")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print("""
Read the curve:
  * gap = 0 (model has no opinion) -> loss = 0.693 = ln 2. The same 'coin flip'
    baseline you met with cross-entropy in Module 5.1.
  * gap strongly positive -> loss -> 0. The model agrees with the human.
  * gap negative -> loss grows without bound. The model actively disagrees.

Training just pushes the gap positive, pair by pair.""")

## 2. A reward model *is* your language model, with one part swapped

Here is the part that surprises people: you don't design a new architecture.

A reward model is your GPT with the **language-model head removed and replaced by a
head that outputs a single number**. Everything else — embeddings, RoPE, every decoder
block, the final norm — is the model you already built and trained.

| | Language model | Reward model |
|---|---|---|
| Trunk | your decoder stack | **the same decoder stack** |
| Head | `Linear(d_model, vocab_size)` | `Linear(d_model, 1)` |
| Output | a distribution over next tokens | one scalar for the whole sequence |
| Read from | every position | the **last** position only |
| Loss | cross-entropy | Bradley–Terry |

Why read the last position? Because in a causal model, the final token is the only one
that has attended to the entire sequence. Its hidden state is the model's summary of
everything it just read — so that's where we hang the score.

And why start from a *trained* LM rather than random weights? Because judging text
requires understanding text. Pretraining already bought that; the score head just needs
to learn which direction in that representation means "good".

In [ ]:
import os
import torch.nn as nn
from llm_workout.model import GPT

torch.manual_seed(0)
ckpt_path = "../checkpoints/tiny_shakespeare.pt"

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    base = GPT(**ckpt["config"])
    base.load_state_dict(ckpt["model_state"])
    stoi, itos = ckpt["stoi"], ckpt["itos"]
    print(f"Loaded YOUR capstone model ({sum(p.numel() for p in base.parameters()):,} params).")
else:
    chars = sorted(set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ ,.!?;:'-\n"))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    base = GPT(vocab_size=len(chars), d_model=128, num_layers=4, num_heads=4,
               hidden_dim=512, max_seq_len=512)
    print("No checkpoint found -- using an UNTRAINED model. Run Module 5.3 first!")

encode = lambda s: [stoi[ch] for ch in s if ch in stoi]
decode = lambda l: "".join(itos[int(i)] for i in l)


class RewardModel(nn.Module):
    """Your GPT, with the vocabulary head swapped for a single-number score head."""

    def __init__(self, base_model: GPT):
        super().__init__()
        self.base = base_model
        # The ONLY new parameters: d_model -> 1.
        self.score_head = nn.Linear(base_model.d_model, 1, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        # Re-run the trunk by hand so we can grab hidden states instead of logits.
        x = self.base.token_embedding(idx)
        freqs_cis = self.base.freqs_cis[:T]
        mask = torch.tril(torch.ones(T, T, device=idx.device)).view(1, 1, T, T)
        for layer in self.base.layers:
            x, _ = layer(x, mask=mask, freqs_cis=freqs_cis)
        x = self.base.final_norm(x)               # (B, T, d_model)
        # Score from the LAST position: the only one that has seen the whole sequence.
        return self.score_head(x[:, -1, :]).squeeze(-1)    # (B,)


rm = RewardModel(base)
new_params = sum(p.numel() for p in rm.score_head.parameters())
print(f"\nReward model built. New parameters added: {new_params:,} "
      f"({new_params / sum(p.numel() for p in rm.parameters()):.4%} of the total).")
print("Everything else is the language model you already trained.")

### Preference data

Real preference datasets are expensive: pay annotators, show them a prompt and two
model outputs, record which they preferred. InstructGPT used tens of thousands of such
comparisons; open datasets like Anthropic's HH-RLHF are in the same range.

We need something our character-level Shakespeare model can actually judge, so we'll
define "better" as **"is this real Shakespeare, or is it mangled?"** Each pair is a
genuine passage (chosen) against a corrupted version of itself (rejected).

That is a toy preference — but it is *structurally identical* to the real thing:
same pair format, same loss, same training loop. Only the definition of "better"
changes when you swap in human labels.

In [ ]:
import random

text_path = "../tinyshakespeare.txt"
if os.path.exists(text_path):
    corpus = open(text_path, encoding="utf-8").read()
else:
    corpus = ("First Citizen:\nBefore we proceed any further, hear me speak.\n\n"
              "All:\nSpeak, speak.\n\n") * 200

BLOCK = 64
rng = random.Random(0)

def corrupt(s):
    """Three ways to make text worse, chosen at random -- our stand-in for a bad answer."""
    mode = rng.choice(["shuffle", "repeat", "noise"])
    if mode == "shuffle":                       # word salad: right words, wrong order
        w = s.split(" ")
        rng.shuffle(w)
        out = " ".join(w)
    elif mode == "repeat":                      # degenerate looping, a real failure mode
        chunk = s[:8] or "the "
        out = chunk * (len(s) // max(len(chunk), 1) + 1)
    else:                                       # random character corruption
        out = "".join(rng.choice(list(stoi)) if rng.random() < 0.3 else ch for ch in s)
    return out[:len(s)].ljust(len(s))

def make_pairs(n):
    chosen, rejected = [], []
    for _ in range(n):
        i = rng.randrange(0, len(corpus) - BLOCK - 1)
        good = corpus[i:i + BLOCK]
        chosen.append(encode(good)[:BLOCK])
        rejected.append(encode(corrupt(good))[:BLOCK])
    # Keep only pairs that survived encoding at full length (chars outside the vocab).
    keep = [k for k in range(n) if len(chosen[k]) == BLOCK and len(rejected[k]) == BLOCK]
    return (torch.tensor([chosen[k] for k in keep]),
            torch.tensor([rejected[k] for k in keep]))

train_chosen, train_rejected = make_pairs(600)
val_chosen, val_rejected = make_pairs(200)
print(f"train pairs: {len(train_chosen)}   val pairs: {len(val_chosen)}")

print("\n--- an example pair ---")
print("CHOSEN  :", repr(decode(train_chosen[0].tolist())))
print("REJECTED:", repr(decode(train_rejected[0].tolist())))

In [ ]:
# Train the reward model with the one-line Bradley-Terry loss.
optimizer = torch.optim.AdamW(rm.parameters(), lr=1e-4)
BATCH, STEPS = 16, 120

@torch.no_grad()
def accuracy(chosen, rejected):
    """How often does the RM rank the good passage above the bad one?"""
    rm.eval()
    correct = 0
    for i in range(0, len(chosen), 32):
        rc = rm(chosen[i:i + 32])
        rr = rm(rejected[i:i + 32])
        correct += (rc > rr).sum().item()
    rm.train()
    return correct / len(chosen)

print(f"val accuracy BEFORE training: {accuracy(val_chosen, val_rejected):.1%}  (chance = 50%)\n")

history = []
for step in range(STEPS):
    ix = torch.randint(len(train_chosen), (BATCH,))
    r_chosen = rm(train_chosen[ix])
    r_rejected = rm(train_rejected[ix])

    # ---- THE ENTIRE REWARD-MODEL LOSS ----
    loss = -F.logsigmoid(r_chosen - r_rejected).mean()
    # --------------------------------------

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0)
    optimizer.step()
    history.append(loss.item())

    if (step + 1) % 30 == 0:
        print(f"step {step + 1:>4}  loss {loss.item():.4f}  "
              f"mean gap {(r_chosen - r_rejected).mean().item():+.3f}")

print(f"\nval accuracy AFTER training : {accuracy(val_chosen, val_rejected):.1%}")

plt.figure(figsize=(6, 3.5))
plt.plot(history, alpha=.4)
plt.axhline(0.693, color="red", ls="--", lw=1, label="ln 2 = no opinion")
plt.xlabel("step"); plt.ylabel("Bradley-Terry loss"); plt.legend()
plt.title("Training a reward model on preference pairs")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# Look at what it learned: score some text it has never seen.
rm.eval()
samples = {
    "real Shakespeare": corpus[50_000:50_000 + BLOCK],
    "shuffled words  ": " ".join(rng.sample(corpus[50_000:50_000 + BLOCK].split(" "),
                                            len(corpus[50_000:50_000 + BLOCK].split(" ")))),
    "repeated loop   ": ("the " * 40)[:BLOCK],
    "random letters  ": "".join(rng.choice(list(stoi)) for _ in range(BLOCK)),
}

print(f"{'text':<20}{'reward':>9}")
print("-" * 32)
with torch.no_grad():
    for name, s in samples.items():
        ids = encode(s)[:BLOCK]
        ids = (ids + encode(" ") * BLOCK)[:BLOCK]      # pad if encoding dropped chars
        print(f"{name:<20}{rm(torch.tensor([ids])).item():>9.3f}")

print("""
Remember Section 1: these numbers have no absolute meaning -- only their ORDER does.
The model has learned a direction in its own representation space that corresponds to
'this looks like real text', purely from pairwise comparisons. Nobody ever showed it
a target score.""")

## 3. The RLHF objective — and why it needs a leash

We have $r(x, y)$. The obvious next move is to train the language model (now called the
**policy**, $\pi_\theta$) to produce answers the reward model scores highly:

$$\max_{\theta}\ \mathbb{E}_{y \sim \pi_\theta(\cdot|x)}\big[r(x, y)\big]$$

Do exactly this and it fails — reliably, and in a specific, famous way.

### Reward hacking

Your reward model is not "human preference". It is a **neural network trained on a
finite sample of human preference**, and it is only accurate near the kind of text it
was trained on. Push a policy hard enough to maximize it and the policy will find the
places where the proxy and the truth come apart — text that scores brilliantly and is
worthless. Documented examples: answers that become relentlessly longer, that open with
flattery, that hedge every claim, that repeat the question back.

This isn't a bug in any particular reward model. It's [Goodhart's law](https://en.wikipedia.org/wiki/Goodhart%27s_law):
*when a measure becomes a target, it ceases to be a good measure.*

### The fix: stay near where you started

The pretrained/SFT model is already fluent and sane. We want to *nudge* it toward
higher reward, not let it wander off into whatever degenerate text maxes out the proxy.
So we keep a **frozen copy** of the starting model — the **reference policy**
$\pi_{\text{ref}}$ — and penalize drifting away from it:

$$\max_{\theta}\ \mathbb{E}\big[r(x, y)\big] \;-\; \beta\, \mathbb{D}_{\text{KL}}\!\big[\pi_\theta(y|x)\,\|\,\pi_{\text{ref}}(y|x)\big]$$

Symbol by symbol:

| Symbol | Meaning |
|---|---|
| $\pi_\theta$ | the **policy** — the model being trained |
| $\pi_{\text{ref}}$ | a **frozen copy** of where training started (the SFT model) |
| $r(x,y)$ | the reward model from Section 2 |
| $\mathbb{D}_{\text{KL}}$ | KL divergence: "how far has the policy's distribution moved?" |
| $\beta$ | the **leash length** — how much drift costs |

$\beta$ is the one knob that matters. Small $\beta$ = long leash = high reward and
likely degeneration. Large $\beta$ = short leash = safe and barely changed.

> **These are the two symbols to remember.** $\pi_{\text{ref}}$ and $\beta$ reappear
> in the DPO loss next module, meaning exactly what they mean here. DPO changes how the
> objective is *optimized*, not what it *is*.

Let's watch reward hacking happen, and watch the leash prevent it.

In [ ]:
# A deliberately tiny world so the whole dynamic is visible.
#
# Five possible responses. The reward model is a PROXY: it over-values length, because
# in its training data longer answers happened to be better. True quality differs --
# and only we (the notebook) can see it. The policy never does.

responses    = ["gibberish", "short & good", "medium & good", "long & padded", "very long & empty"]
true_quality = torch.tensor([-1.0,  0.8,  1.0,  0.3, -0.5])   # what we actually want
proxy_reward = torch.tensor([-1.5,  0.9,  1.3,  1.5,  2.0])   # what the RM scores
#                                                       ^^^  the hackable direction

# The reference (SFT) policy is mediocre: it still puts real mass on gibberish.
# There is genuine room to improve -- which is the whole point of doing this.
ref_logits = torch.tensor([0.8, 0.6, 0.4, 0.2, 0.0])
ref_probs = torch.softmax(ref_logits, dim=0)

def optimize(beta, steps=600, lr=0.1):
    """Maximize E[reward] - beta * KL(policy || reference)."""
    logits = ref_logits.clone().requires_grad_(True)          # start AT the reference
    opt = torch.optim.SGD([logits], lr=lr)
    for _ in range(steps):
        probs = torch.softmax(logits, dim=0)
        expected_reward = (probs * proxy_reward).sum()
        kl = (probs * (probs.log() - ref_probs.log())).sum()
        loss = -(expected_reward - beta * kl)                 # negate: we maximize
        opt.zero_grad(); loss.backward(); opt.step()
    probs = torch.softmax(logits, dim=0).detach()
    return probs, (probs * proxy_reward).sum().item(), (probs * true_quality).sum().item(), \
           (probs * (probs.log() - ref_probs.log())).sum().item()

print(f"{'beta':>7}{'KL':>8}{'proxy reward':>15}{'TRUE quality':>15}   most likely response")
print("-" * 78)
print(f"{'(ref)':>7}{0.0:>8.2f}{(ref_probs * proxy_reward).sum():>15.3f}"
      f"{(ref_probs * true_quality).sum():>15.3f}   {responses[ref_probs.argmax()]}")
for beta in [3.0, 1.5, 0.5, 0.1, 0.0]:
    p, pr, tq, kl = optimize(beta)
    print(f"{beta:>7.2f}{kl:>8.2f}{pr:>15.3f}{tq:>15.3f}   {responses[p.argmax()]}")

print("""
Follow the TRUE quality column down the rows: 0.076 at the reference, rising to ~0.30,
then falling off a cliff to -0.48. The proxy column only ever goes UP.""")

In [ ]:
# The same story as a picture: proxy reward and true quality diverge as the leash loosens.
betas = [4.0, 3.0, 2.0, 1.5, 1.0, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05, 0.0]
proxy_vals, true_vals, kls = [], [], []
for b in betas:
    _, pr, tq, kl = optimize(b)
    proxy_vals.append(pr); true_vals.append(tq); kls.append(kl)

ref_true = (ref_probs * true_quality).sum().item()
peak = max(range(len(betas)), key=lambda k: true_vals[k])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(kls, proxy_vals, "o-", label="proxy reward (what we optimize)")
ax.plot(kls, true_vals, "s-", label="TRUE quality (what we want)")
ax.axhline(ref_true, color="grey", ls=":", lw=1, label="reference policy")
ax.axvline(kls[peak], color="green", ls="--", lw=1,
           label=f"best true quality (beta={betas[peak]})")
ax.set_xlabel("KL from the reference policy  ->  longer leash")
ax.set_ylabel("value")
ax.set_title("Reward hacking: the two curves come apart")
ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"""
Read the gap.

Left of the green line (KL up to ~{kls[peak]:.2f}) both curves rise together. This is the real
improvement RLHF buys, and it is genuine: the policy is moving mass off 'gibberish'
and onto answers that are actually good.

Right of it, the two curves separate. The proxy keeps climbing all the way to
{max(proxy_vals):.2f} while true quality collapses to {min(true_vals):.2f} -- WORSE than the
reference policy we started from. The policy has stopped getting better and started
exploiting its grader.

You cannot see this from inside training. The proxy reward -- the only number the
training loop can measure -- looks better and better the whole way. That is what makes
reward hacking dangerous, and why the KL term is not optional.

In practice: monitor KL as a first-class metric, and tune beta so you sit at the knee.""")

## 4. Why PPO, and why it's painful

We have an objective. Now: how do you actually optimize it?

Not with plain backprop. The thing we're maximizing is an expectation over
$y \sim \pi_\theta$ — **text the model samples itself**. Sampling is not
differentiable: you cannot backpropagate through `torch.multinomial`. This is a
**reinforcement learning** problem, where the "action" is generating a sequence and
the "reward" arrives only at the end.

The standard tool is **PPO** (Proximal Policy Optimization). The loop:

```mermaid
flowchart LR
    A["prompt"] --> B["policy generates<br/>a response"]
    B --> C["reward model<br/>scores it"]
    B --> D["reference model<br/>computes KL"]
    C --> E["value model<br/>estimates baseline"]
    D --> E
    E --> F["PPO update<br/>to the policy"]
    F --> B
```

Count the models in that diagram. **Four**, all in memory at once:

| Model | Role | Trained? |
|---|---|---|
| **Policy** $\pi_\theta$ | the model you're improving | ✅ |
| **Reference** $\pi_{\text{ref}}$ | frozen start point, for the KL term | ❌ frozen |
| **Reward model** $r$ | scores generated text | ❌ frozen |
| **Value model** | estimates expected reward, to reduce gradient variance | ✅ |

For a 70B policy that is a genuinely serious amount of GPU memory — and the generation
step inside the training loop makes it slow, because every update needs fresh samples.
Add RL's usual sensitivity to hyperparameters and you have a pipeline that works
extremely well and is *hard to get right*.

That difficulty is the entire motivation for the next module.

> **RLHF is not obsolete.** It's still what frontier labs use for their flagship
> alignment work, and the whole 2024–25 reasoning-model wave (o1-style models, GRPO,
> RL with verifiable rewards) is RL on LLMs — this loop, with the human-trained reward
> model swapped for an automatic verifier. What DPO offers is a much cheaper path to
> most of the benefit.

## 5. The bridge to DPO

Here is the punchline, and it's worth pausing on.

Someone asked: for a given reward model, what is the policy that *optimally* solves

$$\max_\theta\ \mathbb{E}[r(x,y)] - \beta\,\mathbb{D}_{\text{KL}}[\pi_\theta \| \pi_{\text{ref}}]$$

The answer has a closed form:

$$\pi^*(y|x) = \frac{1}{Z(x)}\,\pi_{\text{ref}}(y|x)\,\exp\!\left(\frac{1}{\beta} r(x,y)\right)$$

Now **run it backwards**. Rearranging for $r$ expresses the reward in terms of the
optimal policy:

$$r(x,y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$$

Substitute *that* into the Bradley–Terry loss from Section 1. The awkward $Z(x)$ term
is identical for both answers to the same prompt, so it **cancels in the difference** —
and you are left with a plain supervised loss over preference pairs, with no reward
model, no sampling, and no RL.

That is **DPO**, and that is Module 6.4. When you meet its loss:

$$\mathcal{L}_{\text{DPO}} = -\log\sigma\!\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)$$

you should recognise every piece:

| Piece | Where you met it |
|---|---|
| $-\log\sigma(\cdot)$ | Section 1 — the Bradley–Terry loss, unchanged |
| the difference inside | chosen minus rejected, exactly as in Section 1 |
| $\pi_{\text{ref}}$ | Section 3 — the frozen reference that stops drift |
| $\beta$ | Section 3 — the leash length |
| $\beta\log\frac{\pi_\theta}{\pi_{\text{ref}}}$ | **the reward model, replaced by the policy itself** |

DPO didn't remove the reward model because rewards were the wrong idea. It removed it
because the algebra showed the language model was secretly one all along.

## Summary

| Idea | What it does |
|---|---|
| **Preference data** | humans can't write the ideal answer, but can pick the better of two |
| **Bradley–Terry** | $P(y_w \succ y_l) = \sigma(r_w - r_l)$ — turns comparisons into scores |
| **Reward model** | your LM with the vocab head swapped for `Linear(d_model, 1)` |
| **Scores are relative** | only gaps mean anything; the absolute scale is arbitrary |
| **Reward hacking** | optimize a proxy hard enough and it stops tracking the truth |
| **The KL leash** | $\beta\,\mathbb{D}_{\text{KL}}[\pi_\theta\|\pi_{\text{ref}}]$ keeps the policy sane |
| **PPO** | sampling isn't differentiable, so alignment becomes RL — four models, slow |
| **→ DPO** | the same objective, solved analytically into one supervised loss |

You trained a real reward model on your own capstone checkpoint, and you watched a
policy hack its own grader. Next, **Module 6.4** collapses all of it into a single
line of code.

### 🏋️ Try it yourself

1. **Feel the leash.** In the toy world, find the $\beta$ that maximizes **true
   quality** by sweeping finely between 0.02 and 1.0. Then note the problem: in a real
   project you cannot compute that column. What would you actually monitor instead?
2. **Make the proxy honest.** Edit `proxy_reward` so it agrees with `true_quality`
   everywhere. Re-run the sweep. Does the hacking disappear? What does that tell you
   about where the danger actually lives — the algorithm, or the reward model?
3. **A harder preference.** Retrain the reward model with a *subtler* corruption — say,
   swapping only two adjacent characters per passage. Does validation accuracy drop?
   This is a small taste of why real preference data is expensive: the closer the two
   answers, the more comparisons you need.
4. **Score your own generations.** Sample 10 continuations from the capstone model at
   `temperature=0.5` and 10 at `temperature=2.0` (Module 5.5), then score all 20 with
   your reward model. Does it prefer the coherent ones? You've just built the
   evaluation half of an RLHF loop.

In [ ]:
# Task 1 starter: sweep beta finely and find the peak of TRUE quality.
fine_betas = [round(0.05 * k, 3) for k in range(1, 81)]      # 0.05 .. 4.0
results = [(b, *optimize(b)[1:]) for b in fine_betas]        # (beta, proxy, true, kl)

best = max(results, key=lambda r: r[2])
ref_true = (ref_probs * true_quality).sum().item()
print(f"Reference policy  : true quality {ref_true:.3f}")
print(f"Best achievable   : true quality {best[2]:.3f} at beta={best[0]} "
      f"(KL={best[3]:.2f}, proxy={best[1]:.3f})")
print(f"Fully unleashed   : true quality {optimize(0.0)[2]:.3f}  <- worse than doing nothing")

print("\nNow the uncomfortable part: in a real project the 'true quality' column")
print("does not exist. What would you monitor instead? (Hint: look at the KL column,")
print("and re-read what Module 5.6 said about qualitative evaluation.)")